# Build a RAG Agent

Retrieval-Augmented Generation with OpenAI + Chroma

Wire up a working RAG agent end to end: embed a knowledge base with Amazon Titan, store and search it in a local Chroma vector DB, and let a LangGraph ReAct agent answer questions with citations. Edit the knowledge file, hit Run, watch the answers change.

## 

## Welcome to Build a RAG Agent

In this project, you'll:

1. **See an AI make stuff up.** You'll ask an AI like ChatGPT a question about a specific document. Watch it confidently invent an answer, just because it has never seen the document.
2. **Teach the AI to read your document first.** You'll write the code that breaks the document into small chunks, lets the AI look things up before answering, and forces it to stick to what's actually in the document.
3. **Prove the fix works.** You'll ask the same question two ways: once to the bare AI, once to the AI that just read your document. The improved version even tells you which lines of the document it used to answer.

This pattern has a name: **RAG**, short for retrieval-augmented generation. It just means the AI looks up information before it answers, instead of answering from memory.

Open the **Document** tab in the left sidebar to see the source text your AI will be reading. The **Guide** at the top of the sidebar has a longer write-up if you want it.

## 

## Phase 1 of 3: Test the LLM on its own

Before building the pipeline, we establish a reference point. We will ask a question whose answer depends on information the LLM was never trained on, and observe how it responds without any retrieval.

## 1 · Ask a bare LLM

Send a question directly to the LLM with no additional context. Because the answer depends on information outside its training data, it must either decline or guess.

In [ ]:
import os
from langchain_openai import ChatOpenAI

# Familiar OpenAI interface. The API key and gateway URL come from the environment,
# so this is exactly the code you'd write against OpenAI itself.
llm = ChatOpenAI(model="google.gemma-3-4b-it", base_url=os.environ["OPENAI_BASE_URL"])

# A question whose answer is in the document but the LLM has no way to know
# it on its own. The document lists the exact domains TensorTonic's coding
# problems cover; without retrieval the LLM can only guess or decline.
question = "What domains do TensorTonic's coding problems cover?"
print(llm.invoke([("human", question)]).content)

## 

## Phase 2 of 3: Build the retrieval pipeline

You saw what happened above: the bare LLM either refused or made something up. TensorTonic doesn't actually offer most of what it claimed. The model has no way to know our product, so it filled the gap with whatever sounded plausible.

The **Document** on the left is the real source of truth. Now we build the pipeline that forces the LLM to read it before answering: load the document, split it into chunks, embed and store those chunks, retrieve the passages closest to a question, and generate an answer grounded in what was actually retrieved. We will inspect the output of each stage before moving on.

## 2 · Load the document

Production RAG systems ingest PDFs, web pages, or databases. We start with a single text file (editable in the Document panel on the left) to keep the focus on the core pattern. The pipeline that follows works on any text source.

In [ ]:
from langchain_core.documents import Document

text = open("document.txt", encoding="utf-8").read()
doc = Document(page_content=text, metadata={"source": "document.txt"})

print(f"Loaded {len(text):,} characters into 1 document.")

## 

### Concept: chunking and embeddings

A document is too long to feed to an LLM whole, and we want to search by meaning, not by exact words. So we do two things:

1. **Chunk.** Split the document into smaller passages of a few hundred characters each, with a small overlap between neighbours so a sentence isn't cut in half.
2. **Embed.** Convert each chunk into a **vector**: a list of numbers (1024 of them for the model we use) that encodes what the chunk means. Think of the vector as the chunk's address in "meaning space". Two pieces of text with similar meaning end up at nearby addresses; unrelated text ends up far apart. We store every chunk's vector in **Chroma**, a small local vector database, so we can search by meaning later.

Reference: [RecursiveCharacterTextSplitter](https://python.langchain.com/docs/how_to/recursive_text_splitter/), [OpenAIEmbeddings](https://python.langchain.com/docs/integrations/text_embedding/openai/), [Chroma](https://python.langchain.com/docs/integrations/vectorstores/chroma/).

## 3 · Chunk and embed (you write this)

Your turn. Split the document into chunks, embed each chunk into a vector, and store the vectors in Chroma so we can search them later. The TODOs guide you. Hints below if you get stuck.

In [ ]:
import os
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO 1: create a splitter.
#   - Use RecursiveCharacterTextSplitter.
#   - Pick a chunk size around 512 characters and an overlap around 64.
#   - Docs: https://python.langchain.com/docs/how_to/recursive_text_splitter/
splitter = None  # replace with your splitter

# TODO 2: turn the loaded `doc` into a list of chunks.
#   - Use splitter.split_documents on a list containing your one Document.
chunks = []  # replace with your chunks

# The embeddings client. Familiar OpenAI interface, pointed at the gateway.
# Each chunk becomes a 1024-dim vector. (check_embedding_ctx_length=False sends
# raw text, not pre-tokenized ids.)
embeddings = OpenAIEmbeddings(model="amazon.titan-embed-text-v2:0", base_url=os.environ["OPENAI_BASE_URL"], check_embedding_ctx_length=False)

# TODO 3: build a Chroma vector store from your chunks.
#   - Use Chroma.from_documents.
#   - Docs: https://python.langchain.com/docs/integrations/vectorstores/chroma/
vector_store = None  # replace with your Chroma store

print(f"{len(chunks)} chunks embedded and indexed.")

## 

### Concept: retrieval

Now that every chunk is a vector, searching becomes: turn the question into a vector too, then find the chunks whose vectors are closest to the question vector.

"Closest" is measured by **cosine distance** in Chroma. Lower distance means more similar.

You typically pull the top **k** chunks (3 to 5 is a strong default). Pulling too many often makes the LLM lose focus.

Reference: [similarity_search_with_score](https://python.langchain.com/docs/integrations/vectorstores/chroma/).

## 4 · Inspect retrieval (you write retrieve)

Write the function that pulls the top-k most similar chunks for a query. Lower distance score means closer match. We also plot the chunks in 2D so you can see where they cluster and where the question lands.

In [ ]:
import io, base64
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Patches plt.show() to send PNG to the TT frontend. Don't touch.
def _tt_show(*args, **kwargs):
    fig = plt.gcf()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=80)
    plt.close(fig)
    print(f"__TT_PNG_START__{base64.b64encode(buf.getvalue()).decode()}__TT_PNG_END__")
plt.show = _tt_show

# TODO: implement retrieve.
#   - Inputs: a query string, and k (number of top chunks to return).
#   - Use vector_store.similarity_search_with_score(query, k=k).
#   - Return whatever similarity_search_with_score returns.
#   - Docs: https://python.langchain.com/docs/integrations/vectorstores/chroma/
def retrieve(query: str, k: int = 3):
    pass  # replace with your implementation

hits = retrieve(question)

# Top hits as a simple bar chart in text.
print(f"Top {len(hits)} hits for: {question!r}\n")
max_d = max(s for _, s in hits)
for i, (chunk, score) in enumerate(hits, 1):
    filled = int(40 * (1 - score / max_d))
    bar = "█" * filled + "░" * (40 - filled)
    print(f"[{i}] {bar} {score:.3f}")
    print(f"    {chunk.page_content[:900].replace(chr(10), ' ')}...\n")

# Where chunks live in embedding space, with your question as a red star.
raw = vector_store._collection.get(include=["embeddings", "documents"])
chunk_vecs = np.array(raw["embeddings"])
q_vec = np.array(embeddings.embed_query(question))
hit_texts = {c.page_content for c, _ in hits}
hit_mask = np.array([t in hit_texts for t in raw["documents"]])

joint = np.vstack([chunk_vecs, q_vec])
centered = joint - joint.mean(axis=0)
_, _, Vt = np.linalg.svd(centered, full_matrices=False)
proj = centered @ Vt[:2].T

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(proj[:-1][~hit_mask, 0], proj[:-1][~hit_mask, 1],
           s=60, c="#666", alpha=0.55, label="chunk")
ax.scatter(proj[:-1][hit_mask, 0], proj[:-1][hit_mask, 1],
           s=110, c="#10b981", edgecolor="black", linewidth=0.6, label="retrieved")
ax.scatter(proj[-1, 0], proj[-1, 1],
           s=240, c="#ef4444", marker="*", edgecolor="black", linewidth=0.6, label="question")
ax.set_title("Chunks in embedding space")
ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
ax.legend(loc="best", fontsize=9, frameon=False)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 

### Concept: grounded prompting

You found relevant chunks. Now you need to make the LLM **use only those chunks** instead of making something up. The request to the model has two pieces, each doing a different job:

| Piece | What it does | Example |
| --- | --- | --- |
| **System prompt** | Sets the rules the model has to follow | "Answer using only the provided context. Cite passages as [chunk N]. Say 'I do not know' if the context does not contain the answer." |
| **User message** | Bundles the retrieved chunks plus the question into one prompt | "Context:\n[chunk 1] ...\n[chunk 2] ...\n\nQuestion: What domains do TensorTonic's coding problems cover?" |

Reference: [ChatOpenAI](https://python.langchain.com/docs/integrations/chat/openai/). LangChain accepts messages as tuples: `("system", "...")`, `("human", "...")`.

## 5 · Generate a grounded answer (you write this)

Now wire retrieval into the LLM. You write the SYSTEM prompt that forces the model to use only the chunks and cite them, plus the ask_with_rag function that runs the full pipeline for one query.

In [ ]:
# TODO 1: write the SYSTEM prompt.
#   - Tell the model to answer using ONLY the provided context.
#   - Tell it to cite passages as [chunk N].
#   - Tell it what to do when the context does not contain the answer.
#   - Tell it to reply in plain text (no markdown).
SYSTEM = ""  # replace with your prompt

# TODO 2: implement ask_with_rag.
#   - Call retrieve(query, k=3) to get the top hits.
#   - Build a context string: each chunk labelled "[chunk N]" then the chunk text.
#   - Build messages: a system message with SYSTEM, then a human message that
#     includes the context AND the question.
#   - Call llm.invoke(messages) and return the .content string.
#   - Docs: https://python.langchain.com/docs/integrations/chat/bedrock/
def ask_with_rag(query: str) -> str:
    pass  # replace with your implementation

print(ask_with_rag(question))

## 

## Phase 3 of 3: Compare and explore

Now we verify the pipeline actually helps. We will ask the same question with and without retrieval to see the difference directly, then try a few follow-ups to find where the pipeline holds up and where it breaks down.

## 6 · Bare LLM vs RAG, side by side

Run the same question two ways: once with no retrieval, once with the RAG pipeline. The bare LLM has no grounding to draw from. The RAG version cites the specific passages it used. The contrast between the two answers is the value retrieval adds.

In [ ]:
def ask_bare(query: str) -> str:
    return llm.invoke([("human", query)]).content

# Same question we asked in cell 1, but this time we run it both ways.
print("BARE LLM (no retrieval)")
print("-" * 40)
print(ask_bare(question))
print()
print("WITH RAG (retrieves from document.txt)")
print("-" * 40)
print(ask_with_rag(question))

## 7 · Ask follow-ups

Try several follow-up questions. When the document does not contain the answer, the model should decline rather than invent one. To extend what the agent knows, edit the Document on the left and re-run cell 3 to re-index.

In [ ]:
# Each question targets a specific detail in the document. The third one
# asks something NOT in the document, so RAG should refuse cleanly.
for q in [
    "What math topics does TensorTonic teach?",
    "What approach does TensorTonic's Research Papers section take?",
    "When was TensorTonic founded?",  # not in doc
]:
    print(f"Q: {q}")
    print(f"A: {ask_with_rag(q)}")
    print()